# Building Data Generation

This notebook shows how to generate building data files for different buildings, locations, weather years, internal gain profiles, and time windows.

Each saved file contains simulated building sensor/state data, controller output, heat-pump/comfort metrics, and the disturbances used for the simulation. Disturbances are generated internally but are not saved as standalone files.

## Setup

Weather loading is handled by `src.disturbances.load_weather`.

- If a matching local DWD TRY file exists in `data/weather` and `year=2015` is requested, it is used locally.
- Otherwise, cached PVGIS JSON files are used when available.
- If no cache exists, PVGIS is queried online and the response is cached in `data/weather`.

For the bundled Freiburg coordinates, `year=2015` uses the local TRY2015 file, while other years fall back to PVGIS for year-specific weather.

I do not see closest-weather-station fallback logic in the current `disturbances.py`; the current fallback is PVGIS. Random city locations are available through `disturbances.get_random_location`, exposed here as `location="random_DE"`.

For uncached locations or years, this notebook needs network access.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

# This notebook lives in notebooks/, so the repo root is one level up.
I4B_ROOT = Path.cwd().resolve()
if I4B_ROOT.name == "notebooks":
    I4B_ROOT = I4B_ROOT.parent

if str(I4B_ROOT) not in sys.path:
    sys.path.insert(0, str(I4B_ROOT))

# The notebook-local plotting helpers live in notebooks/util.py, which is
# importable regardless of whether the kernel starts in notebooks/ or the repo root.
NOTEBOOK_DIR = I4B_ROOT / "notebooks"
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))

import data.buildings as building_catalog
from src import data_generation as dg
from util import plot_building_data

OUTPUT_DIR = I4B_ROOT / "data" / "generated" / "building_data"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Repo root: {I4B_ROOT}")
print(f"Output directory: {OUTPUT_DIR}")

## Available Inputs

In [ ]:
available_buildings = list(building_catalog.__all__)
available_profiles = sorted((I4B_ROOT / "data" / "profiles" / "InternalGains").glob("*.csv"))

print(f"{len(available_buildings)} buildings")
print(available_buildings)
print("\nInternal gain profiles")
for profile in available_profiles:
    print("-", profile.name)

Define locations, weather years, and the export time window. A location can be the building's default location, an explicit override, or a random city sampled by `disturbances.get_random_location` through names such as `random_DE`.

The time window is interpreted as `[START_DATE, END_DATE)`: start is included, end is excluded. This makes one-week or one-month ranges easy to express, for example `2015-01-01` to `2015-02-01`.

In [ ]:
LOCATIONS = {
    **dg.DEFAULT_LOCATIONS,
    # Add project-specific locations here.
    "custom_example": {
        "lat": 52.52,
        "long": 13.405,
        "altitude": 34,
        "timezone": "Europe/Berlin",
    },
}

YEARS = [2015]
TIMESTEP_SECONDS = 900
START_DATE = "2015-01-01"
END_DATE = "2015-01-08"

# Use one of LOCATIONS keys, "building_default", or strings like "random_DE".
LOCATION = "freiburg"

LOCATIONS

## Module Helpers

The reusable implementation lives in `src/data_generation.py`. The notebook calls `dg.generate_building_data_file(...)`, which creates one CSV and one metadata JSON file for a single scenario.

The CSV includes:

- simulated building states such as `T_room`, `T_wall`, `T_hp_ret`, depending on the selected model
- controller output `T_hp_sup`
- disturbances `T_amb`, `Qdot_gains`, `Qdot_sol`, `Qdot_int`
- comfort and heat-pump metrics such as `E_el`, `P_el`, `COP`, `Qdot_th`, `dev_neg_max`
- scenario columns such as building, location, year, profile, method, and controller

In [ ]:
help(dg.generate_building_data_file)

## Controller Configuration

`ctrl_method` selects the controller that produces the heat-pump supply temperature `T_hp_sup`:

| `ctrl_method` | Principle | Keyword arguments |
| --- | --- | --- |
| `"heatcurve"` | feed-forward heating curve, supply temperature is a function of `T_amb` only (no room-temperature feedback) | `T_room_set` [°C], `shift` [K] on the heating limit temperature, `offset` [K] on the nominal supply/return temperatures |
| `"pid"` | feedback PI(D) on the room temperature | `T_room_set` [°C], `KP`, `KI`, `KD` |

Both controllers pass through the same heating-system parameters of the building afterwards
(`src/simulator.py`): `T_hp_sup = max(u + T_offset, T_hp_ret)` while `T_amb < T_amb_lim`, then
`Heatpump.check_hp` clips the request to the heat-pump power range. `building_overrides` lets you change
those building-side parameters without editing `data/buildings/`.

**Two things worth knowing before you interpret a trajectory:**

1. **`T_offset` sets the level of the heating curve** and is stored per building. For
   `sfh_1984_1994_1_enev` the catalog value is `-5` K, which undersupplies the `4R3C` model: the room
   settles around 15–16 °C, roughly 3 K below the comfort bound. The repo's own tuner,
   `src.controller.heatcurve.heatcurve.tune_building_t_offset_and_mdot_hp(...)`, suggests `T_offset = +1` K
   with `mdot_hp = 0.25` for this building, which puts the room at ~20 °C.
2. **The heat pump has a minimum power** (`Q_HP_min = 2000` W in `Heatpump.check_hp`) and no hysteresis.
   Below that request the heat pump is switched off entirely for one step. The corresponding minimum
   temperature lift is `Q_HP_min / (mdot_hp * c_water)` ≈ 2.2 K. Whenever the controller setpoint sits within
   that band above `T_hp_ret`, the simulation alternates between off and exactly 2 kW at every timestep —
   this is the fast oscillation seen with the undersupplied heating curve, not a numerical artifact. Raising
   the curve (or using a coarser timestep) moves the operating point away from that boundary.

Note that `T_room_set_lower` (the night-setback comfort band) is only used to score comfort deviations. The
heating curve never sees it, and the PID tracks the constant `T_room_set` instead.

In [ ]:
BUILDING = "sfh_1984_1994_1_enev"
METHOD = "4R3C"

# Pick the controller: "heatcurve" (feed-forward) or "pid" (room-temperature feedback).
CTRL_METHOD = "heatcurve"

CTRL_KWARGS_BY_METHOD = {
    # Heating curve: T_sup(T_amb). shift moves the heating limit temperature,
    # offset raises/lowers the nominal supply and return temperatures.
    "heatcurve": {"T_room_set": 20.0, "shift": 0.0, "offset": 0.0},
    # PID on the room temperature. The output is a supply temperature, so KP is
    # in K/K; KI is applied to the raw error sum (not scaled by the timestep).
    "pid": {"T_room_set": 20.0, "KP": 10.0, "KI": 0.1, "KD": 0.0},
}
CTRL_KWARGS = CTRL_KWARGS_BY_METHOD[CTRL_METHOD]

# Building-side heating-system parameters the controller output passes through.
# Set to {} to keep the catalog values of data/buildings/.
# T_offset = +1.0 / mdot_hp = 0.25 are the values suggested by
# heatcurve.tune_building_t_offset_and_mdot_hp for this building with 4R3C;
# the catalog value T_offset = -5 leaves the room ~3 K below the comfort bound.
BUILDING_OVERRIDES = {"T_offset": 1.0, "mdot_hp": 0.25}

catalog_params = getattr(building_catalog, BUILDING)
print(f"controller: {CTRL_METHOD} {CTRL_KWARGS}")
print(f"catalog:    T_offset={catalog_params['T_offset']}, "
      f"mdot_hp={catalog_params['mdot_hp']}, T_amb_lim={catalog_params['T_amb_lim']}")
print(f"used:       {BUILDING_OVERRIDES or 'catalog values'}")

# Minimum temperature lift the heat pump can deliver before it switches off.
mdot_hp_used = BUILDING_OVERRIDES.get("mdot_hp", catalog_params["mdot_hp"])
print(f"min. lift:  {2000 / (mdot_hp_used * 4180):.2f} K "
      f"(Q_HP_min = 2000 W at mdot_hp = {mdot_hp_used} kg/s)")

In [ ]:
# Optional: re-tune the heating-system parameters of the selected building with the
# helper that ships with the heating curve. It sweeps T_offset (and then mdot_hp)
# over January and keeps the value whose mean room temperature is closest to 20 degC.
# Takes a few minutes, so it is off by default.
RUN_TUNER = False

if RUN_TUNER:
    import copy

    from src.controller.heatcurve.heatcurve import tune_building_t_offset_and_mdot_hp

    tuned_t_offset, tuned_mdot_hp = tune_building_t_offset_and_mdot_hp(
        copy.deepcopy(catalog_params),
        method=METHOD,
        t_offset_min=-10,
        t_offset_max=20,
        t_offset_step=1.0,
        mdot_hp_min=0.15,
        mdot_hp_max=0.35,
        mdot_hp_step=0.02,
        repo_filepath=f"{I4B_ROOT}/",
    )
    print(f"suggested: T_offset={tuned_t_offset}, mdot_hp={tuned_mdot_hp}")
    print(f"catalog:   T_offset={catalog_params['T_offset']}, mdot_hp={catalog_params['mdot_hp']}")

## Generate One Building Data File

In [ ]:
csv_path, metadata_path, dataset, metadata, results = dg.generate_building_data_file(
    building_name=BUILDING,
    location=LOCATION,
    year=2015,
    profile_name="ResidentialDetached.csv",
    method=METHOD,
    timestep_seconds=900,
    start_date=START_DATE,
    end_date=END_DATE,
    hp_model_name="Heatpump_AW",
    ctrl_method=CTRL_METHOD,
    initial_temperature=20.0,
    night_setback=True,
    output_dir=OUTPUT_DIR,
    locations=LOCATIONS,
    repo_filepath=I4B_ROOT,
    building_overrides=BUILDING_OVERRIDES,
    **CTRL_KWARGS,
)

print(csv_path)
print(metadata_path)
display(dataset.head())
display(dataset.describe())

Inspect the scenario that was just generated directly from the returned `dataset`, without reading the CSV back. `plot_building_data` (in `notebooks/util.py`) draws two stacked panels over a shared time axis:

- **top — external disturbances:** ambient temperature (left axis, °C) plus solar and internal gains (right axis, W)
- **bottom — building and heat pump:** room, supply and return temperature with the room setpoint (left axis, °C) plus heat-pump thermal power (right axis, kW)

Because of the twin axes the vertical position of two curves in one panel is not comparable — read each curve against the axis drawn in its own color. Columns that a given RC model or heat-pump setup does not produce are skipped automatically.

In [ ]:
fig, axes = plot_building_data(dataset)
plt.show()

### Compare Controller Settings

Runs the same week with a few controller configurations and scores them, so the effect of the heating-curve
level and of the controller choice is visible in numbers instead of only in the plot:

- `T_room mean/min` — how well the comfort band is held
- `dev_mean` — mean comfort violation of the `T_room_set_lower` band [K]
- `cycles/day` — heat-pump on/off transitions, i.e. how much the trajectory chatters
- `duty` — share of timesteps with the heat pump running
- `E_el` — electricity demand for the week [kWh]

These runs are written to a separate subdirectory so they do not overwrite the scenario file generated above.

In [ ]:
import numpy as np

COMPARISON_DIR = OUTPUT_DIR / "controller_comparison"


def score_run(data, timestep_seconds=900):
    """Comfort, cycling and energy metrics of one simulated building dataset."""
    room = data["T_room"].astype(float)
    deviation = (data["T_room_set_lower"].astype(float) - room).clip(lower=0)
    running = data["Qdot_th"].astype(float).to_numpy() > 500  # heat pump on
    cycles = int(np.abs(np.diff(running.astype(int))).sum() / 2)
    days = len(data) * timestep_seconds / 86400
    return {
        "T_room mean": round(room.mean(), 1),
        "T_room min": round(room.min(), 1),
        "dev_mean [K]": round(deviation.mean(), 2),
        "dev_max [K]": round(deviation.max(), 2),
        "cycles/day": round(cycles / days, 1),
        "duty [%]": round(running.mean() * 100),
        "E_el [kWh]": round(data["E_el"].astype(float).sum() / 1000, 1),
    }


def run_variant(label, ctrl_method="heatcurve", overrides=None, **ctrl_kwargs):
    _, _, data, _, _ = dg.generate_building_data_file(
        building_name=BUILDING,
        location=LOCATION,
        year=2015,
        profile_name="ResidentialDetached.csv",
        method=METHOD,
        timestep_seconds=900,
        start_date=START_DATE,
        end_date=END_DATE,
        hp_model_name="Heatpump_AW",
        ctrl_method=ctrl_method,
        initial_temperature=20.0,
        night_setback=True,
        output_dir=COMPARISON_DIR,
        locations=LOCATIONS,
        repo_filepath=I4B_ROOT,
        building_overrides=overrides,
        **ctrl_kwargs,
    )
    return label, data


variants = [
    run_variant("heatcurve, catalog T_offset"),
    run_variant("heatcurve, T_offset=+1", overrides={"T_offset": 1.0, "mdot_hp": 0.25}),
    run_variant("heatcurve, offset=+11 K", offset=11.0),
    run_variant("pid KP=10, KI=0.1", ctrl_method="pid", KP=10.0, KI=0.1, KD=0.0),
]

comparison = pd.DataFrame(
    {label: score_run(data) for label, data in variants}
).T
display(comparison)

# Side-by-side view of the two heating-curve levels.
for label, data in variants[:2]:
    fig, axes = plot_building_data(data, title=label)
    plt.show()

## Generate A Scenario Matrix

Adjust these lists to create a dataset matrix. Every scenario saves one building-data CSV and one metadata JSON. If a weather/location/year combination is not cached, the first run may download PVGIS data.

For random locations, set `SCENARIO_LOCATIONS = ["random_DE"]`. The existing random-location function samples a city from Overpass; weather then follows the current `load_weather` behavior.

In [ ]:
SCENARIO_BUILDINGS = [
    "sfh_1979_1983_0_soc",
    "sfh_1984_1994_1_enev",
    "sfh_2002_2009_2_kfw",
]
SCENARIO_LOCATIONS = [LOCATION]
SCENARIO_PROFILES = ["ResidentialDetached.csv", "Office.csv"]
SCENARIO_YEARS = [2015]

# Building-side overrides are per building, so the single-scenario value from
# above is not reused here. Keep the catalog values, or fill in tuned ones, e.g.
# SCENARIO_OVERRIDES = {"sfh_1984_1994_1_enev": {"T_offset": 1.0, "mdot_hp": 0.25}}
SCENARIO_OVERRIDES = {}

saved = []
for building_name in SCENARIO_BUILDINGS:
    for location in SCENARIO_LOCATIONS:
        for profile_name in SCENARIO_PROFILES:
            for year in SCENARIO_YEARS:
                csv_path, metadata_path, dataset, metadata, _ = dg.generate_building_data_file(
                    building_name=building_name,
                    location=location,
                    year=year,
                    profile_name=profile_name,
                    method=METHOD,
                    timestep_seconds=3600,
                    start_date=START_DATE,
                    end_date=END_DATE,
                    hp_model_name="Heatpump_AW",
                    ctrl_method=CTRL_METHOD,
                    initial_temperature=20.0,
                    night_setback=True,
                    output_dir=OUTPUT_DIR,
                    locations=LOCATIONS,
                    repo_filepath=I4B_ROOT,
                    building_overrides=SCENARIO_OVERRIDES.get(building_name),
                    **CTRL_KWARGS,
                )
                saved.append(metadata | {"csv_path": str(csv_path), "metadata_path": str(metadata_path)})

summary = pd.DataFrame(saved)
summary

## Inspect Sensor And Disturbance Columns

The saved file includes building states and the disturbances used by the simulator. The exact state columns depend on the selected RC model.

In [12]:
disturbance_columns = ["T_amb", "Qdot_gains", "Qdot_sol", "Qdot_int"]
sensor_columns = [
    col for col in dataset.columns
    if col.startswith("T_") and col not in disturbance_columns + ["T_room_set_lower"]
]
energy_columns = ["P_el", "E_el", "COP", "Qdot_th"]
comfort_columns = ["dev_neg_sum", "dev_neg_max", "dev_pos_sum", "dev_pos_max"]

display(dataset[sensor_columns + disturbance_columns + energy_columns + comfort_columns].head(24))

,T_room,T_wall,T_hp_ret,T_hp_sup,T_amb,Qdot_gains,Qdot_sol,Qdot_int,P_el,E_el,COP,Qdot_th,dev_neg_sum,dev_neg_max,dev_pos_sum,dev_pos_max
2015-01-01 01:00:00,18.658031,19.453418,19.702501,20.000000,1.3,0.000000,0.000000,0.00,20.860215,20.860215,5.436,113.396126,0.666643,1.341969,0.0,0.0
2015-01-01 02:00:00,18.175464,18.933211,19.297764,19.702501,1.9,0.000000,0.000000,0.00,36.85588,36.85588,5.55023,204.558626,1.566676,1.824536,0.0,0.0
2015-01-01 03:00:00,17.803807,18.444483,18.89282,19.297764,2.4,0.000000,0.000000,0.00,36.833922,36.833922,5.6671,208.741504,1.986888,2.196193,0.0,0.0
2015-01-01 04:00:00,17.418243,17.972404,18.49431,18.892820,2.4,0.000000,0.000000,0.00,37.032403,37.032403,5.721321,211.874259,2.374676,2.581757,0.0,0.0
2015-01-01 05:00:00,17.044098,17.518917,18.101947,18.494310,2.5,0.000000,0.000000,0.00,35.868006,35.868006,5.78787,207.599365,2.753384,2.955902,0.0,0.0
2015-01-01 06:00:00,16.700623,17.092022,17.721489,18.101947,2.9,0.000000,0.000000,0.00,33.366231,33.366231,5.892867,196.622761,3.106348,3.299377,0.0,0.0
2015-01-01 07:00:00,16.352777,16.68207,17.350661,17.721489,3.0,0.000000,0.000000,0.00,32.859176,32.859176,5.958146,195.779754,3.458114,3.647223,0.0,0.0
2015-01-01 08:00:00,16.757989,16.293261,17.156457,17.350661,2.8,697.340000,0.000000,697.34,18.807839,18.807839,5.983309,112.533106,3.410158,3.647223,0.0,0.0
2015-01-01 09:00:00,18.244144,15.959127,17.378073,17.156457,2.9,2133.373652,41.353652,2092.02,0.0,0.0,6.02334,0.0,2.431619,3.242011,0.0,0.0
2015-01-01 10:00:00,18.842231,15.687712,17.769731,17.378073,3.6,2326.240568,234.220568,2092.02,0.0,0.0,6.084191,0.0,1.468321,1.755856,0.0,0.0


## Load A Saved Building Data File

In [ ]:
loaded = pd.read_csv(csv_path, index_col="datetime", parse_dates=True)
print(loaded.shape)
display(loaded.head())

## Inspect A Loaded Building Data File

The same helper works on a file read back from disk — the scenario columns in the CSV supply the figure title.

In [ ]:
fig, axes = plot_building_data(loaded)
plt.show()